In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/task5/train_clean_5_1.csv")
val_df = pd.read_csv("../data/processed/task5/validation_with_listing_type.csv")

print("Train Shape:", train_df.shape)
print("Validation Shape:", val_df.shape)

print("\nTrain Columns:")
print(train_df.columns.tolist())

print("\nValidation Columns:")
print(val_df.columns.tolist())

Train Shape: (5109, 66)
Validation Shape: (1252, 66)

Train Columns:
['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'is_phone_listing', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known', 'storage_imp_src', 'storage_imp_conf', 'storage_imp_post', 'ram_imp_src', 'ram_imp_conf', 'ram_imp_post', 'imp_price_used', 'ref_matched', 'storage_gb_orig', 'storage_gb_was_imputed', 'ram_gb_orig', 'ram_gb_was_imputed', 'ref_n', 'ref_med_log', 'ref_mad_log', 'ref_level', 'ref_median_price', 'price

In [5]:
# 1. Target distribution
print("Train - Listing Type:")
print(train_df['listing_type'].value_counts(dropna=False))

print("\nValidation - Listing Type:")
print(val_df['listing_type'].value_counts(dropna=False))


# 2. Target price statistics
print("\nTrain - Price Statistics:")
print(train_df['price'].describe())

print("\nValidation - Price Statistics:")
print(val_df['price'].describe())


# 3. Missing values
print("\nMissing Prices:")
print("Train:", train_df['price'].isna().sum())
print("Validation:", val_df['price'].isna().sum())


# 4. Duplicate IDs
print("\nDuplicate Ad IDs:")
print("Train:", train_df['ad_id'].duplicated().sum())
print("Validation:", val_df['ad_id'].duplicated().sum())


# 5. Data types
print("\nData Types:")
print(train_df.dtypes.value_counts())

Train - Listing Type:
listing_type
phone            4732
unclassified      366
parts               6
wanted              2
feature_phone       2
tablet              1
Name: count, dtype: int64

Validation - Listing Type:
listing_type
phone           1155
unclassified      97
Name: count, dtype: int64

Train - Price Statistics:
count      5109.000000
mean      24234.327853
std       23642.128663
min        1000.000000
25%        7000.000000
50%       15000.000000
75%       33000.000000
max      152000.000000
Name: price, dtype: float64

Validation - Price Statistics:
count      1252.000000
mean      23562.972843
std       20686.812755
min        1000.000000
25%        8000.000000
50%       16500.000000
75%       34000.000000
max      185500.000000
Name: price, dtype: float64

Missing Prices:
Train: 0
Validation: 0

Duplicate Ad IDs:
Train: 0
Validation: 0

Data Types:
str        27
bool       24
float64    14
int64       1
Name: count, dtype: int64


In [6]:
# Check overlap between training and validation data
print("Overlapping Ad IDs:",
      len(set(train_df['ad_id']) & set(val_df['ad_id'])))

print("Overlapping Title Groups:",
      len(set(train_df['title_group'].dropna()) &
          set(val_df['title_group'].dropna())))


# Inspect potentially price-related features
price_related_cols = [
    'price_missing',
    'price_suspicious',
    'price_very_high',
    'imp_price_used',
    'ref_matched',
    'ref_n',
    'ref_med_log',
    'ref_mad_log',
    'ref_level',
    'ref_median_price',
    'price_ratio',
    'mad_z',
    'price_review_reason',
    'price_flag',
    'price_context'
]

print("\nPrice-related columns:")
display(train_df[price_related_cols].head(5))

Overlapping Ad IDs: 0
Overlapping Title Groups: 0

Price-related columns:


,price_missing,price_suspicious,price_very_high,imp_price_used,ref_matched,ref_n,ref_med_log,ref_mad_log,ref_level,ref_median_price,price_ratio,mad_z,price_review_reason,price_flag,price_context
0,False,False,False,True,True,371.0,9.998798,0.146603,model_key,22000.0,0.886364,-0.554991,NaN,False,NaN
1,False,False,False,False,False,5.0,11.034890,0.016000,model_key,62000.0,1.290323,1.719248,NaN,False,NaN
2,False,False,False,False,True,NaN,NaN,NaN,none,NaN,NaN,NaN,NaN,False,exchange
3,False,False,False,False,False,NaN,NaN,NaN,none,NaN,NaN,NaN,NaN,False,NaN
4,False,False,False,False,False,NaN,NaN,NaN,none,NaN,NaN,NaN,NaN,False,NaN


In [7]:
# Target
target = 'price'

# Columns that must not be used as features
excluded_cols = [
    # Target
    'price',

    # Identifiers and URLs
    'ad_id',
    'ad_url',
    'run_id',
    'title_group',

    # Raw text - we'll use structured information first
    'title',
    'title_normalized',
    'title_punct_normalized',

    # Price-related information (potential leakage)
    'price_missing',
    'price_suspicious',
    'price_very_high',
    'imp_price_used',
    'ref_matched',
    'ref_n',
    'ref_med_log',
    'ref_mad_log',
    'ref_level',
    'ref_median_price',
    'price_ratio',
    'mad_z',
    'price_review_reason',
    'price_flag',
    'price_context',

    # Listing classification is used for filtering, not prediction
    'listing_type',
    'listing_type_rule'
]

feature_cols = [
    col for col in train_df.columns
    if col not in excluded_cols
]

print("Number of features:", len(feature_cols))
print("\nSelected Features:")
print(feature_cols)

# Check feature types
categorical_cols = train_df[feature_cols].select_dtypes(
    include=['object', 'str', 'category']
).columns.tolist()

numerical_cols = train_df[feature_cols].select_dtypes(
    include=['number', 'bool']
).columns.tolist()

print("\nCategorical Features:", len(categorical_cols))
print(categorical_cols)

print("\nNumerical Features:", len(numerical_cols))
print(numerical_cols)

Number of features: 41

Selected Features:
['location', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'is_phone_listing', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known', 'storage_imp_src', 'storage_imp_conf', 'storage_imp_post', 'ram_imp_src', 'ram_imp_conf', 'ram_imp_post', 'storage_gb_orig', 'storage_gb_was_imputed', 'ram_gb_orig', 'ram_gb_was_imputed']

Categorical Features: 16
['location', 'brand', 'storage_src', 'condition', 'condition_src', 'ram_src', 'loc_area', 'loc_city', 'lg_model', 'model_key', 'match_rule', 'family', 'storage_imp_src', 'storage_imp_conf', 'ram_imp_src', 'ram_imp_conf']

Numerical Features: 25
['storage_gb', 'ram_gb', 'has_

In [8]:
# Remove potentially price-derived feature
feature_cols.remove('price_placeholder')

# Check missing values
missing_check = pd.DataFrame({
    'Train Missing': train_df[feature_cols].isna().sum(),
    'Validation Missing': val_df[feature_cols].isna().sum()
})

print("Missing Values:")
display(missing_check[missing_check.sum(axis=1) > 0])


# Check constant features
constant_cols = [
    col for col in feature_cols
    if train_df[col].nunique(dropna=False) <= 1
]

print("\nConstant Features:")
print(constant_cols)


# Check categorical values
categorical_cols = train_df[feature_cols].select_dtypes(
    include=['object', 'str', 'category']
).columns.tolist()

print("\nCategorical Features:", len(categorical_cols))
print(categorical_cols)


# Check unseen categorical values in Validation
for col in categorical_cols:
    train_values = set(train_df[col].dropna().astype(str))
    val_values = set(val_df[col].dropna().astype(str))

    unseen = val_values - train_values

    if unseen:
        print(f"{col}: {len(unseen)} unseen categories")

Missing Values:


,Train Missing,Validation Missing
location,4,2
brand,276,71
storage_gb,994,182
ram_gb,1038,191
storage_src,4720,1155
condition,4754,1175
condition_src,4754,1175
ram_src,4968,1216
loc_area,4,2
loc_city,4,2



Constant Features:
['is_wanted_ad', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'is_feature_phone_hint']

Categorical Features: 16
['location', 'brand', 'storage_src', 'condition', 'condition_src', 'ram_src', 'loc_area', 'loc_city', 'lg_model', 'model_key', 'match_rule', 'family', 'storage_imp_src', 'storage_imp_conf', 'ram_imp_src', 'ram_imp_conf']
location: 87 unseen categories
loc_area: 82 unseen categories
loc_city: 8 unseen categories
model_key: 83 unseen categories
match_rule: 22 unseen categories
family: 3 unseen categories


In [9]:
# Remove feature with almost no information
feature_cols = [
    col for col in feature_cols
    if col != 'lg_model'
]

# Identify categorical features
categorical_cols = train_df[feature_cols].select_dtypes(
    include=['object', 'str', 'category']
).columns.tolist()

# Prepare copies of features
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

# Handle missing categorical values
for col in categorical_cols:
    X_train[col] = X_train[col].fillna('__MISSING__').astype(str)
    X_val[col] = X_val[col].fillna('__MISSING__').astype(str)

# Prepare target
y_train = train_df['price'].copy()
y_val = val_df['price'].copy()

print("Train Features:", X_train.shape)
print("Validation Features:", X_val.shape)

print("Categorical Features:", len(categorical_cols))
print("Missing values in categorical Train:",
      X_train[categorical_cols].isna().sum().sum())

print("Missing values in categorical Validation:",
      X_val[categorical_cols].isna().sum().sum())

print("\nTarget Shapes:")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

Train Features: (5109, 39)
Validation Features: (1252, 39)
Categorical Features: 15
Missing values in categorical Train: 0
Missing values in categorical Validation: 0

Target Shapes:
y_train: (5109,)
y_val: (1252,)


In [10]:
# Inspect feature values and their relationship with the target
derived_cols = [
    'storage_imp_src',
    'storage_imp_conf',
    'storage_imp_post',
    'ram_imp_src',
    'ram_imp_conf',
    'ram_imp_post',
    'confidence',
    'match_rule',
    'ref_matched',
    'storage_gb',
    'ram_gb'
]

for col in derived_cols:
    print(f"\n--- {col} ---")
    print(train_df[col].value_counts(dropna=False).head(10))


--- storage_imp_src ---
storage_imp_src
price+prior:my_data           2282
NaN                           1084
price+prior:ref_base+slope     584
price+prior:ref_price          305
title                          299
price+prior:jumia_price        162
ref_unique                     154
ref_unique_single_source       119
price+prior:none               118
price+prior:my_data+slope        2
Name: count, dtype: int64

--- storage_imp_conf ---
storage_imp_conf
low       1987
medium    1585
NaN       1084
known      299
high       154
Name: count, dtype: int64

--- storage_imp_post ---
storage_imp_post
NaN      1084
1.000     453
0.860     171
0.800     126
0.651      95
0.750      93
0.756      93
0.562      92
0.520      71
0.834      60
Name: count, dtype: int64

--- ram_imp_src ---
ram_imp_src
ref_unique_per_model        2110
ref_popularity              1322
NaN                         1084
ref_unique_given_storage     343
ref_unique_single_source     155
title                         95

In [12]:
# 1. Start with original feature columns
excluded_cols = [
    'price',
    'ad_id', 'ad_url', 'run_id', 'title_group',
    'title', 'title_normalized', 'title_punct_normalized',
    'listing_type', 'listing_type_rule', 'lg_model',

    # Price-related features
    'price_missing', 'price_suspicious', 'price_very_high',
    'imp_price_used', 'ref_matched', 'ref_n',
    'ref_med_log', 'ref_mad_log', 'ref_level',
    'ref_median_price', 'price_ratio', 'mad_z',
    'price_review_reason', 'price_flag', 'price_context',

    # Price-dependent imputation
    'storage_gb', 'ram_gb',
    'storage_src', 'ram_src',
    'storage_imp_src', 'storage_imp_conf', 'storage_imp_post',
    'ram_imp_src', 'ram_imp_conf', 'ram_imp_post'
]

feature_cols = [
    col for col in train_df.columns
    if col not in excluded_cols
]

# 2. Build Train and Validation features
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

# 3. Prepare targets
y_train = train_df['price'].copy()
y_val = val_df['price'].copy()

# 4. Identify categorical columns
categorical_cols = X_train.select_dtypes(
    include=['object', 'str', 'category']
).columns.tolist()

# 5. Handle missing categorical values
for col in categorical_cols:
    X_train[col] = X_train[col].fillna('__MISSING__').astype(str)
    X_val[col] = X_val[col].fillna('__MISSING__').astype(str)

# 6. Final overview
print("Train Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Number of Features:", len(feature_cols))
print("Categorical Features:", len(categorical_cols))
print("Target:", y_train.name)

Train Shape: (5109, 30)
Validation Shape: (1252, 30)
Number of Features: 30
Categorical Features: 9
Target: price


In [13]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# Define experiments
experiments = {
    "All Listings": (
        X_train,
        y_train,
        X_val,
        y_val
    ),

    "Phone Only": (
        X_train.loc[train_df['listing_type'] == 'phone'],
        y_train.loc[train_df['listing_type'] == 'phone'],
        X_val.loc[val_df['listing_type'] == 'phone'],
        y_val.loc[val_df['listing_type'] == 'phone']
    )
}

results = []
models = {}

# Train and evaluate each experiment
for name, (X_tr, y_tr, X_v, y_v) in experiments.items():

    print(f"\nTraining: {name}")
    print("-" * 40)

    model = CatBoostRegressor(
        iterations=1000,
        depth=8,
        learning_rate=0.05,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        verbose=200
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=categorical_cols,
        eval_set=(X_v, y_v),
        early_stopping_rounds=100,
        verbose=200
    )

    predictions = model.predict(X_v)

    mae = mean_absolute_error(y_v, predictions)

    models[name] = model

    results.append({
        'Experiment': name,
        'Train Samples': len(X_tr),
        'Validation Samples': len(X_v),
        'Best Iteration': model.best_iteration_,
        'Validation MAE': round(mae, 2)
    })

# Final comparison
results_df = pd.DataFrame(results)

print("\nFinal Results:")
display(results_df)


Training: All Listings
----------------------------------------
0:	learn: 16125.2285187	test: 14916.7927232	best: 14916.7927232 (0)	total: 276ms	remaining: 4m 35s
200:	learn: 5360.6992014	test: 7509.4526934	best: 7509.4526934 (200)	total: 19.3s	remaining: 1m 16s
400:	learn: 4676.2887394	test: 7414.7927327	best: 7405.2785705 (376)	total: 38.1s	remaining: 56.9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 7383.995096
bestIteration = 455

Shrink model to first 456 iterations.

Training: Phone Only
----------------------------------------
0:	learn: 16346.6112013	test: 14897.1827178	best: 14897.1827178 (0)	total: 97.6ms	remaining: 1m 37s
200:	learn: 4870.8518536	test: 7097.6413409	best: 7097.6413409 (200)	total: 17.9s	remaining: 1m 11s
400:	learn: 4266.3618956	test: 6958.0876430	best: 6953.7177398 (394)	total: 36s	remaining: 53.8s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 6953.71774
bestIteration = 394

Shrink model to first 395 iterations.

F

,Experiment,Train Samples,Validation Samples,Best Iteration,Validation MAE
0,All Listings,5109,1252,455,7384.00
1,Phone Only,4732,1155,394,6953.72


In [14]:
results_df.to_csv(
    "task_5_3_catboost_results.csv",
    index=False
)

print("Task 5.3 results saved successfully.")

Task 5.3 results saved successfully.
